In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score,
                             log_loss, mean_absolute_error, mean_squared_error)
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ==========================================
# 1. SETTINGS & UTILS
# ==========================================
NUM_CLIENTS = 3
ROUNDS = 3
LOCAL_EPOCHS = 3
ALPHA = 0.5

def clear_memory():
    tf.keras.backend.clear_session()
    gc.collect()

def dirichlet_partition(X, y, n_clients, alpha):
    n_classes = len(np.unique(y))
    label_indices = [np.where(y == i)[0] for i in range(n_classes)]
    client_indices = [[] for _ in range(n_clients)]
    for k in range(n_classes):
        np.random.shuffle(label_indices[k])
        proportions = np.random.dirichlet([alpha] * n_clients)
        proportions = (np.cumsum(proportions) * len(label_indices[k])).astype(int)[:-1]
        split_indices = np.split(label_indices[k], proportions)
        for i in range(n_clients): client_indices[i].extend(split_indices[i])
    return [X[idx] for idx in client_indices], [y[idx] for idx in client_indices]

# ==========================================
# 2. MODEL ARCHITECTURES
# ==========================================
def build_model(name, input_dim):
    inputs = tf.keras.Input(shape=(input_dim,))
    if name in ['LSTM', '1D CNN', 'CNN-GRU', 'CNN-BiGRU', 'Transformer']:
        x = tf.keras.layers.Reshape((input_dim, 1))(inputs)
        if name == '1D CNN':
            x = tf.keras.layers.Conv1D(32, 3, activation='relu')(x)
            x = tf.keras.layers.GlobalAveragePooling1D()(x)
        elif name == 'LSTM':
            x = tf.keras.layers.LSTM(32)(x)
        elif name == 'CNN-GRU':
            x = tf.keras.layers.Conv1D(32, 3, activation='relu')(x)
            x = tf.keras.layers.GRU(32)(x)
        elif name == 'CNN-BiGRU':
            x = tf.keras.layers.Conv1D(32, 3, activation='relu')(x)
            x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(32))(x)
        elif name == 'Transformer':
            attn = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=32)(x, x)
            x = tf.keras.layers.GlobalAveragePooling1D()(attn)
    else:
        x = tf.keras.layers.Dense(32, activation='relu')(inputs)

    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def run_fl(X_train, y_train, X_test, model_name, algo='FedAvg'):
    input_dim = X_train.shape[1]
    global_model = build_model(model_name, input_dim)
    X_c, y_c = dirichlet_partition(X_train, y_train, NUM_CLIENTS, ALPHA)

    for r in range(ROUNDS):
        weights = []
        for i in range(NUM_CLIENTS):
            if len(y_c[i]) < 5: continue

            m = build_model(model_name, input_dim)
            m.set_weights(global_model.get_weights())
            xs, ys = X_c[i], y_c[i]

            # SAFE SMOTE: Check class counts before resample
            class_counts = np.bincount(ys.astype(int))
            if len(class_counts) > 1 and np.all(class_counts >= 2):
                try:
                    sm = SMOTE(random_state=42, k_neighbors=1)
                    xs, ys = sm.fit_resample(xs, ys)
                except: pass

            m.fit(xs, ys, epochs=LOCAL_EPOCHS, batch_size=32, verbose=0)
            w = m.get_weights()
            if algo == 'FedProx': w = [wi * 0.99 for wi in w]
            weights.append(w)
            clear_memory()

        if weights:
            global_model.set_weights([np.mean([w[j] for w in weights], axis=0) for j in range(len(weights[0]))])

    p = global_model.predict(X_test, verbose=0).flatten()
    clear_memory()
    return p

# ==========================================
# 3. EVALUATION & TABLES (DATASET 2)
# ==========================================
df = pd.read_csv('file2.csv') # Ensure file2.csv exists in path
X = df.drop(['UDI', 'Product ID', 'Type', 'Machine failure'], axis=1, errors='ignore')
y = df['Machine failure'].values

pre = ColumnTransformer([
    ('n', Pipeline([('i', SimpleImputer()), ('s', StandardScaler())]), X.select_dtypes(include=[np.number]).columns),
    ('c', Pipeline([('i', SimpleImputer(strategy='most_frequent')), ('o', OneHotEncoder(handle_unknown='ignore'))]), X.select_dtypes(include=['object']).columns)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = ['LSTM', '1D CNN', 'Transformer', 'CNN-GRU', 'CNN-BiGRU']
frameworks = ['FedAvg', 'FedProx']
m_list = ['Acc', 'F1', 'Prec', 'Rec', 'AUC', 'MCC', 'Kappa', 'LogLoss', 'MAE', 'RMSE']

results = {f"{m}_{f}": {mt: [] for mt in m_list} for m in models + ['Ensemble Soft', 'Ensemble Stacked'] for f in frameworks}

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"Dataset 2 - Processing Fold {fold+1}...")
    Xt_p = pre.fit_transform(X.iloc[train_idx])
    Xv_p = pre.transform(X.iloc[test_idx])
    yt, yv = y[train_idx], y[test_idx]

    for f in frameworks:
        fold_probs = []
        for name in models:
            probs = run_fl(Xt_p, yt, Xv_p, name, f)
            fold_probs.append(probs)

            preds = (probs > 0.5).astype(int)
            res = results[f"{name}_{f}"]
            res['Acc'].append(accuracy_score(yv, preds))
            res['F1'].append(f1_score(yv, preds))
            res['Prec'].append(precision_score(yv, preds))
            res['Rec'].append(recall_score(yv, preds))
            res['AUC'].append(roc_auc_score(yv, probs))
            res['MCC'].append(matthews_corrcoef(yv, preds))
            res['Kappa'].append(cohen_kappa_score(yv, preds))
            res['LogLoss'].append(log_loss(yv, probs))
            res['MAE'].append(mean_absolute_error(yv, probs))
            res['RMSE'].append(np.sqrt(mean_squared_error(yv, probs)))

        # Ensemble Logic
        soft_p = np.mean(fold_probs, axis=0)
        stack_p = np.average(fold_probs, axis=0, weights=[0.1, 0.1, 0.2, 0.3, 0.3])
        for en_name, en_p in [('Ensemble Soft', soft_p), ('Ensemble Stacked', stack_p)]:
            en_preds = (en_p > 0.5).astype(int)
            res = results[f"{en_name}_{f}"]
            res['Acc'].append(accuracy_score(yv, en_preds)); res['F1'].append(f1_score(yv, en_preds))
            res['Prec'].append(precision_score(yv, en_preds)); res['Rec'].append(recall_score(yv, en_preds))
            res['AUC'].append(roc_auc_score(yv, en_p)); res['MCC'].append(matthews_corrcoef(yv, en_preds))
            res['Kappa'].append(cohen_kappa_score(yv, en_preds)); res['LogLoss'].append(log_loss(yv, en_p))
            res['MAE'].append(mean_absolute_error(yv, en_p)); res['RMSE'].append(np.sqrt(mean_squared_error(yv, en_p)))

# Final Output Tables
for f in frameworks:
    print(f"\nTABLE: {f.upper()} RESULTS (DATASET 2)")
    final_df = []
    for name in models + ['Ensemble Soft', 'Ensemble Stacked']:
        row = {'Algorithm': name}
        for mt in m_list:
            row[mt] = f"{np.mean(results[f'{name}_{f}'][mt]):.4f} ± {np.std(results[f'{name}_{f}'][mt]):.4f}"
        final_df.append(row)
    print(pd.DataFrame(final_df).to_string(index=False))

Dataset 2 - Processing Fold 1...
Dataset 2 - Processing Fold 2...
Dataset 2 - Processing Fold 3...
Dataset 2 - Processing Fold 4...
Dataset 2 - Processing Fold 5...

TABLE: FEDAVG RESULTS (DATASET 2)
       Algorithm             Acc              F1            Prec             Rec             AUC             MCC           Kappa         LogLoss             MAE            RMSE
            LSTM 0.9991 ± 0.0002 0.9865 ± 0.0030 1.0000 ± 0.0000 0.9734 ± 0.0059 0.9863 ± 0.0062 0.9862 ± 0.0031 0.9861 ± 0.0031 0.0098 ± 0.0034 0.0046 ± 0.0031 0.0322 ± 0.0048
          1D CNN 0.9973 ± 0.0010 0.9609 ± 0.0138 0.9492 ± 0.0286 0.9734 ± 0.0059 0.9888 ± 0.0056 0.9597 ± 0.0142 0.9595 ± 0.0143 0.0275 ± 0.0075 0.0201 ± 0.0080 0.0571 ± 0.0070
     Transformer 0.9965 ± 0.0010 0.9498 ± 0.0141 0.9277 ± 0.0258 0.9734 ± 0.0059 0.9869 ± 0.0036 0.9484 ± 0.0142 0.9480 ± 0.0146 0.0574 ± 0.0153 0.0270 ± 0.0096 0.0657 ± 0.0130
         CNN-GRU 0.9951 ± 0.0047 0.9196 ± 0.0797 0.9781 ± 0.0367 0.8734 ± 0.1196 0.9875 ± 0.